# SparkSQL Window Functions · Olist demosu

*VBO Data Engineering Bootcamp · 8. hafta*

> **İş sorusu:** Müşteri ikinci kez geliyor mu, ne zaman geliyor? Her kategoride hangi satıcılar öne çıkıyor?

Sunumda yöntemleri gördük. Burada tek bir veri setinde **doğru yöntemi seçerek** beş soruyu cevaplıyoruz. Her perdenin ritmi aynı:

1. **İş sorusu**
2. Hangi fonksiyon, `PARTITION BY` ne, `ORDER BY` ne?
3. Sorgu
4. **Sağlama:** sonucu bağımsız bir yoldan doğrulayan küçük kontrol (`assert`)

| Perde | Soru | Fonksiyon |
|---|---|---|
| 1 | Müşterinin kaçıncı siparişi? | `ROW_NUMBER` |
| 2 | İki sipariş arası kaç gün? | `LAG` |
| 3 | Aylık ciro: running total, 3 aylık ortalama | `SUM/AVG OVER` + `ROWS BETWEEN` |
| 4 | Kategori içinde ilk 3 satıcı | `DENSE_RANK` |
| 5 | Harcamaya göre müşteri çeyrekleri | `NTILE(4)` |

## 0. SparkSession ve Olist view'ları

CSV'ler `notebooks/datasets/olist/` altında olmalı (indirme adımı README'de). Her CSV bir temp view olur; bundan sonrası yalnızca `spark.sql`.

In [1]:
from pathlib import Path
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Window Functions · Olist")
    .master("local[*]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")   # varsayılan 200; tek makinede küçük veri için 8 yeter
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")   # WARN logları hücre çıktısını kirletmesin

DATA = Path("datasets/olist")
VIEWS = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "products": "olist_products_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
    "payments": "olist_order_payments_dataset.csv",
}
missing = [f for f in VIEWS.values() if not (DATA / f).exists()]
assert not missing, f"Eksik CSV: {missing}. README > Veriyi indir adımını uygula."

for view, file in VIEWS.items():
    spark.read.csv(str(DATA / file), header=True, inferSchema=True).createOrReplaceTempView(view)

spark.sql("SHOW VIEWS").show()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/05 05:33:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+---------+--------------------+-----------+
|namespace|            viewName|isTemporary|
+---------+--------------------+-----------+
|         |category_translation|       true|
|         |           customers|       true|
|         |         order_items|       true|
|         |              orders|       true|
|         |            payments|       true|
|         |            products|       true|
+---------+--------------------+-----------+



## Perde 1 · Müşterinin kaçıncı siparişi? (`ROW_NUMBER`)

**İş sorusu:** Her siparişe "bu, müşterinin kaçıncı siparişi?" numarasını verelim. Kaç müşteri ikinci kez geliyor?

**Sohbete sor:** Hangi fonksiyon? `PARTITION BY` ne? `ORDER BY` ne?  ⏸ *Bekle, cevapları oku.*

In [2]:
# İlk deneme: müşteri = customer_id
spark.sql("""
    SELECT order_id, customer_id, order_purchase_timestamp,
           ROW_NUMBER() OVER (PARTITION BY customer_id
                              ORDER BY order_purchase_timestamp) AS order_no
    FROM orders
""").createOrReplaceTempView("order_seq_wrong")

spark.sql("SELECT order_no, COUNT(*) AS orders FROM order_seq_wrong GROUP BY order_no").show()

+--------+------+
|order_no|orders|
+--------+------+
|       1| 99441|
+--------+------+



**Herkes 1 çıktı.** Sorgu doğru, partition yanlış: Olist'te `customer_id` her siparişte yeniden üretilir. Gerçek kişiyi `customers` tablosundaki `customer_unique_id` taşır.

In [3]:
# Sağlama: customer_id sipariş başına tekil mi?
spark.sql("""
    SELECT COUNT(*) AS orders, COUNT(DISTINCT customer_id) AS customer_ids FROM orders
""").show()
assert spark.sql("SELECT MAX(order_no) FROM order_seq_wrong").first()[0] == 1

+------+------------+
|orders|customer_ids|
+------+------------+
| 99441|       99441|
+------+------------+



In [4]:
# Düzeltme: partition = customer_unique_id
spark.sql("""
    SELECT c.customer_unique_id, o.order_id, o.order_purchase_timestamp,
           ROW_NUMBER() OVER (PARTITION BY c.customer_unique_id
                              ORDER BY o.order_purchase_timestamp, o.order_id) AS order_no
    FROM orders o
    JOIN customers c USING (customer_id)
""").createOrReplaceTempView("order_seq")

# En çok sipariş veren müşterinin siparişleri
spark.sql("""
    SELECT * FROM order_seq
    WHERE customer_unique_id = (SELECT customer_unique_id FROM order_seq ORDER BY order_no DESC LIMIT 1)
    ORDER BY order_no
""").show(truncate=False)

+--------------------------------+--------------------------------+------------------------+--------+
|customer_unique_id              |order_id                        |order_purchase_timestamp|order_no|
+--------------------------------+--------------------------------+------------------------+--------+
|8d50f5eadf50201ccdcedfb9e2ac8455|5d848f3d93a493c1c8955e018240e7ca|2017-05-15 23:30:03     |1       |
|8d50f5eadf50201ccdcedfb9e2ac8455|369634708db140c5d2c4e365882c443a|2017-06-18 22:56:48     |2       |
|8d50f5eadf50201ccdcedfb9e2ac8455|5837a2c844decae8a778657425f6d664|2017-07-17 22:11:13     |3       |
|8d50f5eadf50201ccdcedfb9e2ac8455|4f62d593acae92cea3c5662c76122478|2017-07-18 23:10:58     |4       |
|8d50f5eadf50201ccdcedfb9e2ac8455|bf92c69b7cc70f7fc2c37de43e366173|2017-07-24 22:11:50     |5       |
|8d50f5eadf50201ccdcedfb9e2ac8455|519203404f6116d406a970763ee75799|2017-08-05 08:59:43     |6       |
|8d50f5eadf50201ccdcedfb9e2ac8455|e3071b7624445af6e4f3a1b23718667d|2017-09-05 22:1

In [5]:
# İş cevabı: tekrar gelen müşteri oranı
repeat = spark.sql("""
    SELECT COUNT(DISTINCT customer_unique_id)                              AS customers,
           COUNT(DISTINCT CASE WHEN order_no = 2 THEN customer_unique_id END) AS repeat_customers,
           ROUND(100 * repeat_customers / customers, 2)                    AS repeat_pct
    FROM order_seq
""")
repeat.show()

# Sağlama: aynı sayı window'suz, GROUP BY ile
by_group = spark.sql("""
    SELECT COUNT(*) FROM (
        SELECT c.customer_unique_id FROM orders o JOIN customers c USING (customer_id)
        GROUP BY c.customer_unique_id HAVING COUNT(*) > 1)
""").first()[0]
r = repeat.first()
assert r.repeat_customers == by_group
assert 2.5 < r.repeat_pct < 3.5

+---------+----------------+----------+
|customers|repeat_customers|repeat_pct|
+---------+----------------+----------+
|    96096|            2997|      3.12|
+---------+----------------+----------+



**İş cevabı:** 96 bin müşterinin yalnızca **~%3'ü** ikinci kez gelmiş. Sorgu boş değil, cevap bu: Olist bir pazar yeri ve müşterilerin büyük kısmı tek seferlik. Sadakat ya da "ikinci sipariş" kampanyası konuşulacaksa başlangıç noktası bu oran.

### Spark UI'da bak

Window sorgusu Spark'ta iki operatöre dönüşür: **`Exchange hashpartitioning(customer_unique_id)`** (aynı müşterinin satırları aynı partition'a taşınır, yani shuffle) ve **`Window`**. Canlıda: http://localhost:4040 → **SQL / DataFrame** sekmesi → son sorgu. Aşağıdaki hücre aynı planı metin olarak gösterir.

In [6]:
plan = spark.sql("""
    EXPLAIN SELECT customer_unique_id,
                   ROW_NUMBER() OVER (PARTITION BY customer_unique_id ORDER BY order_purchase_timestamp) AS order_no
            FROM orders JOIN customers USING (customer_id)
""").first()[0]
print(plan)
assert "Window" in plan and "Exchange hashpartitioning(customer_unique_id" in plan

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [customer_unique_id#43, order_no#308]
   +- Window [row_number() windowspecdefinition(customer_unique_id#43, order_purchase_timestamp#20 ASC NULLS FIRST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS order_no#308], [customer_unique_id#43], [order_purchase_timestamp#20 ASC NULLS FIRST]
      +- Sort [customer_unique_id#43 ASC NULLS FIRST, order_purchase_timestamp#20 ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(customer_unique_id#43, 8), ENSURE_REQUIREMENTS, [plan_id=1871]
            +- Project [customer_unique_id#43, order_purchase_timestamp#20]
               +- BroadcastHashJoin [customer_id#18], [customer_id#42], Inner, BuildRight, false
                  :- Filter isnotnull(customer_id#18)
                  :  +- FileScan csv [customer_id#18,order_purchase_timestamp#20] Batched: false, DataFilters: [isnotnull(customer_id#18)], Format: CSV, Location: InMemoryFileIndex(1 

## Perde 2 · İki sipariş arası kaç gün? (`LAG`)

**İş sorusu:** Tekrar gelen müşteri, bir önceki siparişinden kaç gün sonra geliyor?

**Sohbete sor:** Hangi fonksiyon? `PARTITION BY` ne? `ORDER BY` ne?  ⏸ *Bekle, cevapları oku.*

In [7]:
spark.sql("""
    WITH x AS (
        SELECT c.customer_unique_id, o.order_id, o.order_purchase_timestamp AS ts,
               LAG(o.order_purchase_timestamp) OVER (PARTITION BY c.customer_unique_id
                                                     ORDER BY o.order_purchase_timestamp, o.order_id) AS prev_ts
        FROM orders o
        JOIN customers c USING (customer_id)
    )
    SELECT *, DATEDIFF(ts, prev_ts) AS days_since_prev FROM x
""").createOrReplaceTempView("order_gaps")

spark.sql("SELECT * FROM order_gaps WHERE prev_ts IS NOT NULL ORDER BY customer_unique_id, ts").show(8, truncate=False)

+--------------------------------+--------------------------------+-------------------+-------------------+---------------+
|customer_unique_id              |order_id                        |ts                 |prev_ts            |days_since_prev|
+--------------------------------+--------------------------------+-------------------+-------------------+---------------+
|00172711b30d52eea8b313a7f2cced02|c306eca42d32507b970739b5b6a5a33a|2018-08-13 09:14:07|2018-07-28 00:23:49|16             |
|004288347e5e88a27ded2bb23747066c|08204559bebd39e09ee52dcb56d8faa2|2018-01-14 07:36:54|2017-07-27 14:13:03|171            |
|004b45ec5c64187465168251cd1c9c2f|9392c5e72885ad5aba87e6223ca9838d|2018-05-26 19:42:48|2017-09-01 12:11:23|267            |
|0058f300f57d7b93c477a131a59b36c3|81a93b2fa39e104b865b2bc471c16008|2018-03-22 18:09:41|2018-02-19 17:11:34|31             |
|00a39521eb40f7012db50455bf083460|cea3e6c11eb60acb9d8d4d51694832f8|2018-06-03 10:12:57|2018-05-23 20:14:21|11             |
|00cc12a

İlk siparişin öncesi yok: `LAG` orada `NULL` döner. Yalnız tekrar gelenlere bakmak için `prev_ts IS NOT NULL` yeterli.

In [8]:
gaps = spark.sql("""
    SELECT COUNT(*)                                          AS repeat_orders,
           MEDIAN(days_since_prev)                           AS median_days,
           ROUND(AVG(days_since_prev), 1)                    AS avg_days,
           ROUND(100 * AVG(IF(days_since_prev = 0, 1, 0)), 1) AS same_day_pct,
           MEDIAN(IF(days_since_prev > 0, days_since_prev, NULL)) AS median_days_excl_same_day
    FROM order_gaps
    WHERE prev_ts IS NOT NULL
""")
gaps.show()

# Sağlama: LAG'ı dolu satır sayısı = sipariş sayısı - tekil müşteri sayısı
g = gaps.first()
expected = spark.sql("""
    SELECT COUNT(*) - COUNT(DISTINCT c.customer_unique_id) FROM orders o JOIN customers c USING (customer_id)
""").first()[0]
assert g.repeat_orders == expected

+-------------+-----------+--------+------------+-------------------------+
|repeat_orders|median_days|avg_days|same_day_pct|median_days_excl_same_day|
+-------------+-----------+--------+------------+-------------------------+
|         3345|       28.0|    78.2|        28.7|                     67.0|
+-------------+-----------+--------+------------+-------------------------+



**İş cevabı:** 3.345 tekrar siparişin **%29'u aynı gün** verilmiş (sepeti bölmek, ikinci bir ödeme); bunlar gerçek "geri dönüş" değil. Aynı günleri çıkarınca medyan **67 gün**: müşteri döndüğünde ~2 ay sonra dönüyor. Ortalama (78) uzak dönüşlerle şişiyor; çarpık dağılımda medyana bak. "İkinci sipariş" hatırlatması için pencere 1-2. ay.

## Perde 3 · Aylık ciro: running total ve 3 aylık hareketli ortalama (`SUM/AVG OVER` + `ROWS BETWEEN`)

**İş sorusu:** Teslim edilen siparişlerde aylık ciro nasıl birikiyor, kısa vadeli eğilim ne?

Tam aylar: 2017-01 … 2018-08. Ciro = `payments.payment_value`.

**Sohbete sor:** Hangi fonksiyon? `PARTITION BY` ne? `ORDER BY` ne?  ⏸ *Bekle, cevapları oku.* *Ve frame ne olmalı?*

In [9]:
monthly = spark.sql("""
    WITH m AS (
        SELECT DATE_TRUNC('MONTH', o.order_purchase_timestamp) AS month,
               SUM(p.payment_value)                            AS revenue
        FROM orders o
        JOIN payments p USING (order_id)
        WHERE o.order_status = 'delivered'
          AND o.order_purchase_timestamp >= '2017-01-01'
          AND o.order_purchase_timestamp <  '2018-09-01'
        GROUP BY 1
    )
    SELECT DATE_FORMAT(month, 'yyyy-MM') AS month,
           revenue,
           SUM(revenue) OVER (ORDER BY month ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_total,
           AVG(revenue) OVER (ORDER BY month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)         AS ma_3m,
           AVG(revenue) OVER (ORDER BY month RANGE BETWEEN INTERVAL 2 MONTHS PRECEDING AND CURRENT ROW) AS ma_3m_range
    FROM m
    ORDER BY month
""")
monthly.selectExpr("month", "CAST(revenue AS BIGINT) AS revenue",
                   "CAST(running_total AS BIGINT) AS running_total",
                   "CAST(ma_3m AS BIGINT) AS ma_3m").show(24)

+-------+-------+-------------+-------+
|  month|revenue|running_total|  ma_3m|
+-------+-------+-------------+-------+
|2017-01| 127545|       127545| 127545|
|2017-02| 271298|       398844| 199422|
|2017-03| 414369|       813213| 271071|
|2017-04| 390952|      1204165| 358873|
|2017-05| 567066|      1771232| 457462|
|2017-06| 490225|      2261458| 482748|
|2017-07| 566403|      2827862| 541232|
|2017-08| 646000|      3473862| 567543|
|2017-09| 701169|      4175032| 637858|
|2017-10| 751140|      4926173| 699436|
|2017-11|1153528|      6079701| 868612|
|2017-12| 843199|      6922900| 915955|
|2018-01|1078606|      8001507|1025111|
|2018-02| 966510|      8968017| 962772|
|2018-03|1120678|     10088695|1055265|
|2018-04|1132933|     11221629|1073374|
|2018-05|1128836|     12350466|1127482|
|2018-06|1012090|     13362557|1091287|
|2018-07|1027903|     14390461|1056277|
|2018-08| 985414|     15375875|1008469|
+-------+-------+-------------+-------+



**`ROWS` mu `RANGE` mı?** `ROWS BETWEEN 2 PRECEDING` son 3 *satırı* alır, `RANGE BETWEEN INTERVAL 2 MONTHS PRECEDING` son 3 *takvim ayını*: bir ay eksik olsaydı (Olist'te 2016-11 yok) `ROWS` penceresi 4 aya yayılırdı. Burada her ay dolu, ikisi aynı sonucu verir.

**`PARTITION BY` yok:** tüm aylar tek sıralamada, Spark bütün veriyi **tek partition'a** toplar (log: `No Partition Defined for Window operation!`). 20 satırda sorun değil; milyarlık tabloda darboğaz.

In [10]:
# Sağlama
rows = monthly.collect()
total = spark.sql("""
    SELECT SUM(p.payment_value) FROM orders o JOIN payments p USING (order_id)
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp >= '2017-01-01' AND o.order_purchase_timestamp < '2018-09-01'
""").first()[0]
assert len(rows) == 20                                                   # 20 ay, boşluk yok
assert abs(rows[-1].running_total - total) < 0.01                        # running total'ın sonu = toplam ciro
assert abs(rows[0].ma_3m - rows[0].revenue) < 0.01                       # ilk ayda pencere tek satır
assert abs(rows[2].ma_3m - sum(r.revenue for r in rows[:3]) / 3) < 0.01  # 3. ay = ilk 3 ayın ortalaması
assert all(abs(r.ma_3m - r.ma_3m_range) < 0.01 for r in rows)            # ay eksik değil: ROWS = RANGE

**İş cevabı:** 2017'de ciro hızla büyüyor; Kasım 2017'deki tepe (Black Friday) tek ayda sıçrama yapıyor, 3 aylık ortalama bunu yumuşatıyor. 2018'de ortalama ~1 milyon BRL/ay bandında yatay.

## Perde 4 · Kategori içinde ilk 3 satıcı (`DENSE_RANK`)

**İş sorusu:** Her kategoride ciroya göre öne çıkan ilk 3 satıcı kim?

**Sohbete sor:** Hangi fonksiyon? `PARTITION BY` ne? `ORDER BY` ne?  ⏸ *Bekle, cevapları oku.*

In [11]:
spark.sql("""
    SELECT COALESCE(t.product_category_name_english, p.product_category_name, 'unknown') AS category,
           i.seller_id,
           SUM(i.price) AS revenue
    FROM order_items i
    JOIN orders o   USING (order_id)
    JOIN products p USING (product_id)
    LEFT JOIN category_translation t USING (product_category_name)
    WHERE o.order_status = 'delivered'
    GROUP BY 1, 2
""").createOrReplaceTempView("seller_revenue")

# Gerçek veride ilk 3 içinde kuruşu kuruşuna eşit ciro var mı?
exact_ties = spark.sql("""
    SELECT category, revenue, COUNT(*) AS sellers
    FROM (SELECT *, DENSE_RANK() OVER (PARTITION BY category ORDER BY revenue DESC) AS rnk FROM seller_revenue)
    WHERE rnk <= 3
    GROUP BY category, revenue
    HAVING COUNT(*) > 1
""")
exact_ties.show()
assert exact_ties.count() == 0

+--------+-------+-------+
|category|revenue|sellers|
+--------+-------+-------+
+--------+-------+-------+



**Not:** Gerçek veride ilk 3 içinde birebir eşit ciro **yok**. Raporda ciroyu **yüz BRL'ye yuvarlıyoruz** (`ROUND(revenue, -2)`); yönetim "9.100 BRL" ile "9.149 BRL"yi zaten aynı görür. Eşitlik bu yuvarlamadan doğuyor, bunu bilerek yapıyoruz.

In [12]:
# İlk deneme: ROW_NUMBER
spark.sql("""
    SELECT category, seller_id, ROUND(revenue, -2) AS revenue_100,
           ROW_NUMBER() OVER (PARTITION BY category ORDER BY ROUND(revenue, -2) DESC) AS rn
    FROM seller_revenue
""").createOrReplaceTempView("top_rn")

spark.sql("SELECT * FROM top_rn WHERE rn <= 4 AND category = 'fashion_male_clothing' ORDER BY rn").show(truncate=False)

+---------------------+--------------------------------+-----------+---+
|category             |seller_id                       |revenue_100|rn |
+---------------------+--------------------------------+-----------+---+
|fashion_male_clothing|aaed1309374718fdd995ee4c58c9dfcd|3600.0     |1  |
|fashion_male_clothing|a3a38f4affed601eb87a97788c949667|2500.0     |2  |
|fashion_male_clothing|dc8798cbf453b7e0f98745e396cc5616|1000.0     |3  |
|fashion_male_clothing|835f0f7810c76831d6c7d24c7a646d4d|1000.0     |4  |
+---------------------+--------------------------------+-----------+---+



4. satırın cirosu 3. ile aynı (1.000 BRL) ama `rn = 4`: `WHERE rn <= 3` onu listeden atar. Eşitlikte hangisinin 3, hangisinin 4 olacağı da belirsiz; her çalıştırmada değişebilir. Eşitlikte adil sıra için `RANK` ya da `DENSE_RANK`.

In [13]:
# Düzeltme: DENSE_RANK
spark.sql("""
    SELECT category, seller_id, ROUND(revenue, -2) AS revenue_100,
           DENSE_RANK() OVER (PARTITION BY category ORDER BY ROUND(revenue, -2) DESC) AS rnk
    FROM seller_revenue
""").createOrReplaceTempView("top_dr")

spark.sql("SELECT * FROM top_dr WHERE rnk <= 3 AND category = 'fashion_male_clothing' ORDER BY rnk").show(truncate=False)

+---------------------+--------------------------------+-----------+---+
|category             |seller_id                       |revenue_100|rnk|
+---------------------+--------------------------------+-----------+---+
|fashion_male_clothing|aaed1309374718fdd995ee4c58c9dfcd|3600.0     |1  |
|fashion_male_clothing|a3a38f4affed601eb87a97788c949667|2500.0     |2  |
|fashion_male_clothing|dc8798cbf453b7e0f98745e396cc5616|1000.0     |3  |
|fashion_male_clothing|835f0f7810c76831d6c7d24c7a646d4d|1000.0     |3  |
+---------------------+--------------------------------+-----------+---+



In [14]:
# Sağlama: eşitlik olan kategoride DENSE_RANK bir satıcı fazla getiriyor; genelde de ROW_NUMBER satıcı kaybediyor
n = lambda q: spark.sql(q).first()[0]
assert n("SELECT COUNT(*) FROM top_rn WHERE rn  <= 3 AND category = 'fashion_male_clothing'") == 3
assert n("SELECT COUNT(*) FROM top_dr WHERE rnk <= 3 AND category = 'fashion_male_clothing'") == 4
lost = n("SELECT COUNT(*) FROM top_dr WHERE rnk <= 3") - n("SELECT COUNT(*) FROM top_rn WHERE rn <= 3")
print(f"ROW_NUMBER ile listeden düşen satıcı sayısı (tüm kategoriler): {lost}")
assert lost > 0

ROW_NUMBER ile listeden düşen satıcı sayısı (tüm kategoriler): 13


### `QUALIFY` Spark'ta var mı?

Databricks, Snowflake, BigQuery gibi motorlarda window sonucu alt sorgusuz filtrelenir: `... QUALIFY rnk <= 3`. **Açık kaynak Spark 4.1.3'te yok**, `PARSE_SYNTAX_ERROR` verir. Bu yüzden yukarıda view/alt sorgu + `WHERE` kullandık.

In [15]:
try:
    spark.sql("""
        SELECT category, seller_id FROM seller_revenue
        QUALIFY DENSE_RANK() OVER (PARTITION BY category ORDER BY revenue DESC) <= 3
    """).show(3)
    qualify_supported = True
except Exception as e:
    print(str(e).strip().splitlines()[0])
    qualify_supported = False
assert not qualify_supported

[PARSE_SYNTAX_ERROR] Syntax error at or near 'DENSE_RANK'. SQLSTATE: 42601 (line 3, pos 16)


**İş cevabı:** Her kategoride öne çıkan satıcılar listesi `top_dr WHERE rnk <= 3`. Eşitlikte `ROW_NUMBER` sessizce satıcı düşürür; satıcıya prim ya da rozet verilecekse bu hata gerçek bir kişiyi dışarıda bırakır.

## Perde 5 · Harcamaya göre müşteri çeyrekleri (`NTILE(4)`)

**İş sorusu:** Müşterileri toplam harcamaya göre dört eşit gruba bölelim. En üst çeyrek cironun ne kadarını getiriyor?

**Sohbete sor:** Hangi fonksiyon? `PARTITION BY` ne? `ORDER BY` ne?  ⏸ *Bekle, cevapları oku.*

In [16]:
spark.sql("""
    WITH spend AS (
        SELECT c.customer_unique_id, SUM(p.payment_value) AS total_spend
        FROM orders o
        JOIN customers c USING (customer_id)
        JOIN payments p  USING (order_id)
        WHERE o.order_status = 'delivered'
        GROUP BY 1
    )
    SELECT *, NTILE(4) OVER (ORDER BY total_spend DESC) AS quartile FROM spend
""").createOrReplaceTempView("spend_q")

quartiles = spark.sql("""
    SELECT quartile,
           COUNT(*)                                                     AS customers,
           ROUND(MIN(total_spend), 2)                                   AS min_spend,
           ROUND(MAX(total_spend), 2)                                   AS max_spend,
           ROUND(100 * SUM(total_spend) / SUM(SUM(total_spend)) OVER (), 1) AS revenue_share_pct
    FROM spend_q
    GROUP BY quartile
    ORDER BY quartile
""")
quartiles.show()

+--------+---------+---------+---------+-----------------+
|quartile|customers|min_spend|max_spend|revenue_share_pct|
+--------+---------+---------+---------+-----------------+
|       1|    23340|   182.56| 13664.08|             59.4|
|       2|    23339|   107.78|   182.55|             21.3|
|       3|    23339|    63.06|   107.78|             12.6|
|       4|    23339|     9.59|    63.05|              6.6|
+--------+---------+---------+---------+-----------------+



Burada da `PARTITION BY` yok: 93 bin müşteri tek partition'da sıralanır (Perde 3'teki uyarı). `SUM(SUM(total_spend)) OVER ()` ise GROUP BY sonucunun üstünde çalışan bir window: boş `OVER ()` = tüm sonuç tek pencere, payın paydası.

In [17]:
# Sağlama: gruplar eşit büyüklükte (en fazla 1 fark), paylar %100 eder, sınırlar örtüşmez
q = quartiles.collect()
assert len(q) == 4
assert max(r.customers for r in q) - min(r.customers for r in q) <= 1
assert abs(sum(r.revenue_share_pct for r in q) - 100) < 0.5
assert all(q[i].min_spend >= q[i + 1].max_spend for i in range(3))

**İş cevabı:** En üst çeyrek (harcaması ~183 BRL üstü) cironun **~%59'unu** getiriyor, en alt çeyrek **~%7'sini**. Tekrar gelen müşteri az (Perde 1) olduğundan bu fark çoğunlukla tek siparişin büyüklüğünden geliyor.

## Kapanış · İş sorusuna cevap

> *Müşteri ikinci kez geliyor mu, ne zaman geliyor? Her kategoride hangi satıcılar öne çıkıyor?*

| Perde | Yöntem | Cevap |
|---|---|---|
| 1 | `ROW_NUMBER`, `PARTITION BY customer_unique_id` | Müşterilerin ~%3'ü ikinci kez geliyor. Partition'ı yanlış anahtarla kurunca herkes "ilk sipariş" görünüyordu. |
| 2 | `LAG` | Tekrar siparişlerin %29'u aynı gün; gerçek dönüşte medyan 67 gün. Hatırlatma penceresi 1-2. ay. |
| 3 | `SUM/AVG OVER` + `ROWS BETWEEN` | 2017'de büyüme, Kasım 2017 tepe, 2018'de ~1 milyon BRL/ay yatay. |
| 4 | `DENSE_RANK` | Her kategorinin ilk 3 satıcısı; `ROW_NUMBER` yüz BRL'ye yuvarlanmış ciroda 13 satıcıyı sessizce düşürüyordu. QUALIFY Spark 4.1.3'te yok. |
| 5 | `NTILE(4)` | En üst çeyrek cironun ~%59'u, en alt çeyrek ~%7'si. |

**Yöntem seçerken sorulacak üç soru:** Satır başına ne üretiyorum (sıra, önceki değer, birikim, grup)? Kimin içinde (`PARTITION BY`)? Hangi sırayla ve hangi pencerede (`ORDER BY`, frame)?

In [18]:
spark.stop()